In [1]:
import qiskit_metal as metal
from qiskit_metal.qlibrary.user_components.reader import *
from qiskit_metal.qlibrary.user_components.writer import *
from qiskit_metal.qlibrary.user_components.save_load import *
from qiskit_metal.qlibrary.user_components.layout import single_qubit_layout,build_airbridge,two_qubit_layout
from qiskit_metal.qlibrary.user_components.my_qcomponent import *
from qiskit_metal import designs
import copy
import json
from qiskit_metal import draw, Dict,designs,MetalGUI
import gdspy

## 1. 参数化设计约瑟夫森结
这里的约瑟夫森结有两种，第一种是十字结型的SQUID，为可调耦合器的结；第二种是十字型单结，为固定频率比特的结；


In [2]:
design = metal.designs.DesignPlanar()
gui = metal.MetalGUI(design)
design.overwrite_enabled = True

### 1.1 结的种类(结文件分Cell设计)
#### 1.1.1 十字型单结
固定频率比特所在的结

In [4]:
design.delete_all_components()
options = Dict(pos_x='0um',
               pos_y='0um',
               JJ_pad_lower_width='8.9um',
               JJ_pad_lower_height='4um',
               JJ_pad_up_width='5.4um',
               JJ_pad_up_height='10um',
               finger_lower_width='0.18um',
               finger_lower_height='8.108um',
               finger_lower_pos='2.3um',
               finger_up_pos='5um',
               finger_up_width='0.18um',
               finger_up_height='8.017um',
               extension='2.6um',
               orientation='0')
jj3 = JJManhattan(design, 'jj_manhattan',options=options)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['jj_manhattan'])

In [5]:
design.delete_all_components()
options = Dict(pos_x='-7.2um',
               pos_y='10.33um',
               JJ_pad_lower_width='6um',
               JJ_pad_lower_height='8.8um',
               JJ_pad_up_width='10um',
               JJ_pad_up_height='5.2um',
               finger_lower_width='0.18um',
               finger_lower_height='7.767um',
               finger_lower_pos='4.9um',
               finger_up_pos='2um',
               finger_up_width='0.18um',
               finger_up_height='8.017um',
               extension='2.6um',
               orientation='90')
jj4 = JJManhattan(design, 'jj_manhattan_flip',options=options)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['jj_manhattan_flip'])

jj_path = ''
a_gds = writer_planar(design,jj_path)
a_gds.export_to_gds('JJ_Manhattan_Flip.gds')

03:11PM 05s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
03:11PM 05s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\pythonProject".


1

#### 1.1.2 十字型SQUID
可调耦合器所在的结

In [6]:
design.delete_all_components()
options = Dict(pos_x='-1.2um',
               pos_y='0um',
               JJ_pad_lower_width='10um',
               JJ_pad_lower_height='6.2um',
               JJ_pad_up_width='4um',
               JJ_pad_up_height='9.0um',
               finger_left_lower_width='0.18um',
               finger_right_lower_width='0.18um',
               finger_lower_height='5.8um',
               finger_up_height='4.372um',
               finger_up_pos='0.923um',
               finger_left_up_width='0.18um',
               finger_right_up_width='0.18um',
               JJ_space='5.6um',
               extension='1.7228um',
               orientation='-90')
jj1 = JJSquid(design,'jj_squid',options=options)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['jj_squid'])

jj_path = ''
a_gds = writer_planar(design,jj_path)
a_gds.export_to_gds('JJ_SQUID.gds')

03:11PM 08s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
03:11PM 08s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\pythonProject".


1

In [7]:
design.delete_all_components()
options = Dict(pos_x='-1.9um',
               pos_y='0um',
               JJ_pad_lower_width='10um',
               JJ_pad_lower_height='5.2um',
               JJ_pad_up_width='4um',
               JJ_pad_up_height='9.0um',
               JJ_pad_up_width2='6um',
               JJ_pad_up_height2='3.4um',
               finger_left_lower_width='0.18um',
               finger_right_lower_width='0.18um',
               finger_lower_height='7.4um',
               finger_up_height='7.732um',
               finger_left_up_width='0.18um',
               finger_right_up_width='0.18um',
               JJ_up_pos='1um',
               JJ_down_pos='2.357um',
               extension='2.45um',
               orientation='-90')
jj2 = JJSquidFlip(design,'jj_squid_flip',options=options)
gui.rebuild()
gui.autoscale()
gui.zoom_on_components(['jj_squid_flip'])

jj_path = ''
a_gds = writer_planar(design,jj_path)
a_gds.export_to_gds('JJ_SQUID_Flip.gds')

03:11PM 10s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
03:11PM 10s WARNING [_import_junction_gds_file]: Not able to find file:"".  Not used to replace junction. Checked directory:"D:\Users\ls\PycharmProjects\pythonProject".


1

### 1.2 结文件
将上述gds分文件以cell的形式整合为结文件以供单比特版图使用,其中一个cell为SQUID，另外一个为单结
### 注：在生成结文件后需要将其放入resources文件夹中，否则无法生效

In [8]:
#结文件，两种不同类型的结
gds1 = gdspy.GdsLibrary(infile='./JJ_Manhattan.gds', unit=1e-03, precision=1e-13)
top_main_cell1 = gds1.cells['TOP_main_1']
gds1.rename_cell(top_main_cell1, 'FakeJunction_01')

gds2 = gdspy.GdsLibrary(infile='./JJ_SQUID.gds', unit=1e-03, precision=1e-13)
top_main_cell2 = gds2.cells['TOP_main_1']
gds2.rename_cell(top_main_cell2, 'FakeJunction_02')

gds3 = gdspy.GdsLibrary(infile='./JJ_Manhattan_Flip.gds', unit=1e-03, precision=1e-13)
top_main_cell3 = gds3.cells['TOP_main_1']
gds3.rename_cell(top_main_cell3, 'FakeJunction_01_r')

gds4 = gdspy.GdsLibrary(infile='./JJ_SQUID_Flip.gds', unit=1e-03, precision=1e-13)
top_main_cell4 = gds4.cells['TOP_main_1']
gds4.rename_cell(top_main_cell4, 'FakeJunction_02_r')

# Create a new GDS library to hold the combined cells
combined_gds = gdspy.GdsLibrary(unit=1e-03, precision=1e-13)
for i in range(1,5):
    combined_gds.add(globals()[f"top_main_cell{i}"])
    
combined_gds.write_gds('Fake_Junctions.GDS')
print("GDS files combined successfully into 'Fake_Junctions.GDS'")

GDS files combined successfully into 'Fake_Junctions.GDS'


## 2.分版图设计
在完成约瑟夫森结文件后，需要设计可调耦合器双比特版图。
### 2.1 初始化版图设计
设计双比特分版图并对其几何参数进行初始化，其中jj_dict是每个比特对应的约瑟夫森结名字字典。

In [2]:
jj_dict = {'Q1':'FakeJunction_01','Q2':'FakeJunction_01','Q3':'FakeJunction_01_r','Q4':'FakeJunction_01_r',
           'tunable_coupler1':'FakeJunction_02_r','tunable_coupler2':'FakeJunction_02',}  
# design,gui= two_qubit_layout(jj_dict)
design,gui = two_qubit_layout(jj_dict=jj_dict, parameter_file_path='data/2qubit/two_qubit_parameter_v2-0106.json', airbridge=False, version='0.00')

  return original_import(name, *args, **kwargs)

10:44AM 04s WARNING [check_lengths]: For path table, component=R6, key=trace has short segments that could cause issues with fillet. Values in (22-23)  are index(es) in shapely geometry.
10:44AM 04s WARNING [check_lengths]: For path table, component=R6, key=cut has short segments that could cause issues with fillet. Values in (22-23)  are index(es) in shapely geometry.


In [3]:
#设置参数文件索引为name，设置结位置
set_index='name'
jj_path = './resources/Fake_Junctions.GDS'
cheese_view = {1: False, 2: False, 3: False, 4: False, }
#设置输出文件选项
a_gds = writer_planar(design,jj_path,negative_mask=[1,2,3],cheese_view=cheese_view)
# a_gds.export_to_gds('small_tunable_coupler_test'+str(1)+'.gds')
a_gds.export_to_gds('two_qubits_tunable_coupler_v'+str(10)+'.gds')


10:44AM 27s WARNING [_give_rotation_center_twopads]: In junction table, component=Q1 with name=rect_jj has width=0.008 smaller than cell dimension=0.014318.
10:44AM 27s WARNING [_give_rotation_center_twopads]: In junction table, component=Q2 with name=rect_jj has width=0.008 smaller than cell dimension=0.014318.
10:44AM 27s WARNING [_give_rotation_center_twopads]: In junction table, component=Q3 with name=rect_jj has width=0.008 smaller than cell dimension=0.017827000000000003.
10:44AM 27s WARNING [_give_rotation_center_twopads]: In junction table, component=Q4 with name=rect_jj has width=0.008 smaller than cell dimension=0.017827000000000003.
10:44AM 27s WARNING [_give_rotation_center_twopads]: In junction table, component=tunable_coupler1 with name=rect_jj has width=0.008 smaller than cell dimension=0.0299.
10:44AM 27s WARNING [_give_rotation_center_twopads]: In junction table, component=tunable_coupler2 with name=rect_jj has width=0.008 smaller than cell dimension=0.024878.


1

### 2.2 读出参数文件存储为JSON文件
在本设计中，参数以excel文件的形式提供并且格式需要是标准化的以供外部读取，具体如下
1. 每一行代表版图中需要改变参数的器件名字，如Q1-Q5， meanderQ1-meanderQ9等，其取名参考初始版图代码中的命名；
2. 每一列代表器件中的几何参数，这里将所有需要改变的器件参数都需要列出来，如果几何参数不属于某个器件，则留白，系统将自动转换为None
   例如 total_length 是谐振腔参数，则在Q1的total_length那一列则留白不填；
3. 几何参数遵循{}嵌套规则，具体如connection_pads{readout{pad_gap}}代表的是比特上和谐振腔耦合处的pad_gap参数，即字典connection_pads:{readout:{pad_gap}}

读取函数将以一定规则读取这个参数文件并将其保存为JSON格式以供后续的参数修改

### 2.3 参数输入
参数输入函数将JSON格式的参数文件导入到初始版图中对分版图参数进行逐一修改

In [4]:
# set_index='name'
# path='data/tunable_coupler_layout2.csv'
# savedpath='data\design'+str(2)+'.json'
# parameter=extract_nested_dict(path,set_index)
# save_json_parameter(parameter,savedpath)
# parameter_import(design,load_json(savedpath))
# gui.rebuild()
# gui.autoscale()

In [ ]:
# path_json = f'data/tunable_coupler_v2_layout_option.json'
# save_json(design,path_json)
# print(f"tunable_coupler_v2_layout_option.json created.")

### 2.4 空气桥生成
在谐振腔和控制线部分生成空气桥

In [5]:
list0=[]
list1=[]
list2=[]
for i in range(1, 7):
    list0.append(design.components['meander'+str(i)])
for i in range(1, 5):
    list1.append(design.components['readout1_part'+str(i)])  
    list1.append(design.components['readout2_part'+str(i)])
for i in range(1,6):
    list2.append(design.components['xy'+str(i)+'_part2'])
list2.append(design.components['z1_part2'])
list2.append(design.components['z2_part2'])
# list2.append(design.components['xy1_part2'])

parameter_dict = Dict(pad_width='22 um', etch_residue='-2 um',
                      bridge_length='50um', pad_layer=3, etch_layer=4)
build_airbridge(design,gui,list_meander=list0,list_controlLine=list2+list1,parameter_dict=parameter_dict) 
# build_airbridge(design,gui,list_meander=None,list_controlLine=list2,) 

# a_gds.options.gds_unit=1
# a_gds.export_to_gds('design'+str(i)+'.gds')
# print(f'gds{i} is finished')

In [6]:
a_gds.export_to_gds('design_tunable_coupler'+str(1)+'.gds')

04:04PM 23s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
04:04PM 23s WARNING [_qgeometry_to_gds]: Unexpected shapely object geometry.The variable qgeometry_element is <class 'numpy.float64'>, method can currently handle Polygon and FlexPath.
04:04PM 23s WARNING [_give_rotation_center_twopads]: In junction table, component=xmon_round1 with name=rect_jj has width=0.008 smaller than cell dimension=0.014318.
04:04PM 23s WARNING [_give_rotation_center_twopads]: In junction table, component=xmon_round2 with name=rect_jj has width=0.008 smaller than cell dimension=0.014318.
04:04PM 23s WARNING [_give_rotation_center_twopads]: In junction table, component=tunable_coupler1 with name=rect_jj has width=0.008 smaller than cell dimension=0.024800000000000003.
04:04PM 23s WARNING [_give_rotation_center_twopads]: In junction table, component=xmon_round3 with name=rect_jj has

1

## 3.总版图生成
### 3.1 加入标志
这里加入标志的方法有两种，第一种是先创建分版图标志然后通过程序控制在总版图的位置；第二种是直接创建总版图标志模版，然后将分版图放入总版图模版中
本设计先采用第二种方法

In [8]:
import klayout.db as db

layout = db.Layout()
layout.read('design_tunable_coupler'+str(1)+'.gds')
db.SaveLayoutOptions.gds2_user_units=0.001
layout.write('design_tunable_coupler'+str(1)+'.gds')

In [10]:
# Load the GDS file
template = gdspy.GdsLibrary(infile='gds/template.gds')
gds = gdspy.GdsLibrary(infile='design_tunable_coupler'+str(1)+'.gds')

# delta_x = 9600
# delta_y = 8000
cell = gds.cells['TOP_main_1']
for layer,polygon in cell.get_polygons(by_spec=True).items():
    polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])   
    template.cells['TOP'].add(polygon_set)
cell = gds.cells['TOP_main_3']
for layer,polygon in cell.get_polygons(by_spec=True).items():
    polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])   
    template.cells['TOP'].add(polygon_set)
cell = gds.cells['TOP_main_4']
for layer,polygon in cell.get_polygons(by_spec=True).items():
    polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])   
    template.cells['TOP'].add(polygon_set)
#     
# cell1 = gds.cells['TOP']
# for layer,polygon in cell1.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell2 = gds.cells['TOP']
# for layer,polygon in cell2.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(0,delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell3 = gds.cells['TOP']
# for layer,polygon in cell3.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(delta_x,delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell4 = gds.cells['TOP']
# for layer,polygon in cell4.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,0)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell5 = gds.cells['TOP']
# for layer,polygon in cell5.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1])       
#     template.cells['TOP'].add(polygon_set)
# 
# cell6 = gds.cells['TOP']
# for layer,polygon in cell6.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(delta_x,0)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell7 = gds.cells['TOP']
# for layer,polygon in cell7.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(-delta_x,-delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell8 = gds.cells['TOP']
# for layer,polygon in cell8.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(0,-delta_y)        
#     template.cells['TOP'].add(polygon_set)
# 
# cell9 = gds.cells['TOP']
# for layer,polygon in cell9.get_polygons(by_spec=True).items():
#     polygon_set = gdspy.PolygonSet(polygon,layer[0],layer[1]).translate(delta_x,-delta_y)        
#     template.cells['TOP'].add(polygon_set)


template.write_gds('tunable_coupler_v4.gds')
print("GDS files combined successfully generated")

GDS files combined successfully generated
